# minGPT 可执行学习 Notebook

这个 Notebook 不是独立教材，而是 `docs/learning_guide.html` 的实验台。

推荐使用方式：

1. 先打开 `docs/learning_guide.html` 阅读图解和逐行解释。
2. 在 HTML 里看到对应实验时，回到本 Notebook 执行 cell。
3. 观察真实输出：shape、loss、logits、生成结果、训练变化。
4. 再回到 HTML 和源码，解释为什么输出是这样。

## HTML ↔ Notebook 对照表

| HTML 章节 | Notebook cell | 你要观察什么 |
| --- | --- | --- |
| 0. 导读与核心思想 | Cell 0-3 | 环境、导入、设备、minGPT 是否可用 |
| 5. 整体架构与数据流 | Cell 4-10 | `gpt-nano` 配置、输入 shape、logits shape |
| 6-8. Attention / Block / GPT | Cell 13-14 + `mingpt/model.py` | hooks 捕获的中间激活，和源码里的模块对应 |
| 13. 训练的本质 | Cell 15-18 | Dataset、Trainer、loss 如何下降 |
| 14. 自回归生成 | Cell 11-12、19-20 | 未训练/训练后生成的差别，为什么是一格一格生成 |
| 训练、反向传播、优化器 | Cell 21-22 | `loss.backward()` 后梯度如何出现 |

注意：HTML 负责“讲清楚为什么”，Notebook 负责“亲眼看到输出”，源码负责“确认真实实现”。普通 HTML 不能直接执行 `.ipynb`，因为执行需要 Python kernel、依赖环境和文件系统权限。



## 第 1 步：导入和环境检查
## Step 1: Imports and Environment Check

In [4]:

# ============ 诊断：检查所有模块是否正确安装 ============
# ============ Diagnosis: Check if all modules are installed correctly ============

print("🔍 检查模块... (Checking modules...)\n")

modules_to_check = [
    ('torch', 'PyTorch - 深度学习框架'),
    ('numpy', 'NumPy - 数值计算'),
    ('regex', 'Regex - 正则表达式（BPE tokenizer 需要）'),
    ('requests', 'Requests - HTTP 库（下载 GPT-2 模型）'),
]

all_ok = True
for module_name, description in modules_to_check:
    try:
        __import__(module_name)
        print(f"✓ {module_name:12} - {description}")
    except ImportError as e:
        print(f"✗ {module_name:12} - 缺失！ | Missing! ({e})")
        all_ok = False

print()
if all_ok:
    print("✓ 所有模块都已安装！(All modules installed successfully!)\n")
else:
    print("⚠️ 有模块缺失，运行以下命令安装：")
    print("⚠️ Some modules are missing. Run this command:")
    print("pip install regex requests -q")
    print()


🔍 检查模块... (Checking modules...)

✓ torch        - PyTorch - 深度学习框架
✓ numpy        - NumPy - 数值计算
✓ regex        - Regex - 正则表达式（BPE tokenizer 需要）
✓ requests     - Requests - HTTP 库（下载 GPT-2 模型）

✓ 所有模块都已安装！(All modules installed successfully!)



In [5]:
import torch
import torch.nn as nn
import numpy as np
from mingpt.model import GPT
from mingpt.trainer import Trainer
from mingpt.bpe import BPETokenizer
from mingpt.utils import set_seed

# 设置随机种子以获得可重现的结果
# 再現可能な結果を得るためにランダムシードを設定します
set_seed(3407)

# 检查 GPU 可用性
# GPU の可用性を確認します
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ 设备: {device}')
print(f'✓ PyTorch 版本: {torch.__version__}')
if device == 'cuda':
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✓ 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

✓ 设备: cpu
✓ PyTorch 版本: 2.7.0+cu118


## 第 2 步：创建一个小型 GPT 模型
## Step 2: Create a Small GPT Model

我们将创建一个「gpt-nano」模型用于学习。这个模型很小，可以快速训练和测试。

GPT-nanoモデルを作成します。このモデルは小さいので、すばやくトレーニングとテストができます。

In [10]:
# 创建模型配置
# モデル設定を作成します
config = GPT.get_default_config()
config.model_type = 'gpt-nano'  # 最小的模型（最小のモデル）
config.vocab_size = 50257  # GPT-2 词汇表大小（GPT-2 vocabulary size）
config.block_size = 1024  # 上下文窗口大小（context window size）
config.device = device

print("模型配置 (Model Configuration):")
print(f"  vocab_size: {config.vocab_size}  # 词汇表大小 (Vocabulary size)")
print(f"  block_size: {config.block_size}  # 上下文窗口 (Context window)")
print(f"  n_layer: {config.n_layer}  # Transformer 块数 (Transformer block count)")
print(f"  n_head: {config.n_head}    # 注意力头数 (Attention head count)")
print(f"  n_embd: {config.n_embd}    # 嵌入维度 (Embedding dimension)")

# 创建模型
# モデルを作成します
model = GPT(config)
model.to(device)

# 统计参数数量
# パラメータ数を統計します
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ 模型创建完成！(Model created successfully!)")
print(f"  总参数数: {total_params:,} (Total parameters)")
print(f"  可训练参数: {trainable_params:,} (Trainable parameters)")

模型配置 (Model Configuration):
  vocab_size: 50257  # 词汇表大小 (Vocabulary size)
  block_size: 1024  # 上下文窗口 (Context window)
  n_layer: None  # Transformer 块数 (Transformer block count)
  n_head: None    # 注意力头数 (Attention head count)
  n_embd: None    # 嵌入维度 (Embedding dimension)
number of parameters: 2.55M

✓ 模型创建完成！(Model created successfully!)
  总参数数: 4,958,736 (Total parameters)
  可训练参数: 4,958,736 (Trainable parameters)


## 第 3 步：理解模型结构
## Step 3: Understanding Model Architecture

让我们检查模型内部的结构。

In [11]:
# 打印模型架构
# モデルアーキテクチャを印刷します
print("模型架构 (Model Architecture):")
print(model)

print("\n" + "="*60)
print("关键组件说明 (Key Components):")
print("="*60)
print("""
1. wte (Word Token Embedding) - 将 token ID 转换为向量
   Convert token IDs to vectors
   
2. wpe (Word Position Embedding) - 添加位置信息
   Add positional information to embeddings
   
3. drop (Dropout) - 正则化技巧
   Regularization technique
   
4. h (Transformer Blocks) - 堆叠的 Transformer 块
   Stack of Transformer blocks
   
5. ln_f (Layer Norm) - 最后的标准化
   Final normalization layer
   
6. lm_head (Language Model Head) - 输出层
   Output projection to vocabulary
""")

# 检查关键层的大小
# 主要な層のサイズを確認します
print("\n关键层的维度 (Dimensions of Key Layers):")
print(f"  Token Embedding 形状: {model.transformer.wte.weight.shape}")
print(f"    (vocab_size=50257, embedding_dim={config.n_embd})")
print(f"  Position Embedding 形状: {model.transformer.wpe.weight.shape}")
print(f"    (max_positions={config.block_size}, embedding_dim={config.n_embd})")

模型架构 (Model Architecture):
GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 48)
    (wpe): Embedding(1024, 48)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-2): 3 x Block(
        (ln_1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=48, out_features=144, bias=True)
          (c_proj): Linear(in_features=48, out_features=48, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=48, out_features=192, bias=True)
          (c_proj): Linear(in_features=192, out_features=48, bias=True)
          (act): NewGELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
  )
  (lm_he

## 第 4 步：前向传播 (Forward Pass)
## Step 4: Forward Pass Visualization

现在让我们看看数据如何通过模型流动。

In [13]:
# 创建随机输入
# ランダム入力を作成します
batch_size = 2
seq_length = 16
x = torch.randint(0, config.vocab_size, (batch_size, seq_length)).to(device)

print(f"输入形状 (Input shape): {x.shape}")
print(f"  batch_size={batch_size}, seq_length={seq_length}")
print(f"输入内容 (Sample input tokens): {x[0, :8].tolist()}")

# 前向传播
# 前向伝播を実行します
print("\n执行前向传播...")
with torch.no_grad():
    output = model(x)
    # 如果输出是元组，取第一个元素（logits）
    # If output is a tuple, take the first element (logits)
    logits = output[0] if isinstance(output, tuple) else output

print(f"✓ 前向传播完成！")
print(f"输出形状 (Logits shape): {logits.shape}")
print(f"  (batch_size={batch_size}, seq_length={seq_length}, vocab_size={config.vocab_size})")

# 获取概率分布
# 確率分布を取得します
probs = torch.softmax(logits, dim=-1)
print(f"\n概率分布 (Probability distribution):")
print(f"  形状: {probs.shape}")
print(f"  最后一个 token 的前 5 个概率最高的 token ID:")
top_probs, top_indices = torch.topk(probs[0, -1], k=5)
for i, (idx, prob) in enumerate(zip(top_indices.tolist(), top_probs.tolist())):
    print(f"    {i+1}. Token {idx}: {prob:.4f}")

输入形状 (Input shape): torch.Size([2, 16])
  batch_size=2, seq_length=16
输入内容 (Sample input tokens): [40590, 48683, 46378, 955, 11255, 13037, 47149, 39771]

执行前向传播...
✓ 前向传播完成！
输出形状 (Logits shape): torch.Size([2, 16, 50257])
  (batch_size=2, seq_length=16, vocab_size=50257)

概率分布 (Probability distribution):
  形状: torch.Size([2, 16, 50257])
  最后一个 token 的前 5 个概率最高的 token ID:
    1. Token 50204: 0.0000
    2. Token 41639: 0.0000
    3. Token 12063: 0.0000
    4. Token 26468: 0.0000
    5. Token 39530: 0.0000


## 第 5 步：文本生成
## Step 5: Text Generation

让我们用模型生成文本。注意：这个模型没有经过训练，所以生成的文本会很随机。

In [14]:
# 准备 tokenizer
# トークナイザーを準備します
tokenizer = BPETokenizer()

# 创建一个空的开始序列
# 空の開始シーケンスを作成します
context = torch.zeros((1, 1), dtype=torch.long).to(device)  # batch_size=1

print("生成文本... (generating text...)")
print("注意：模型还没训练，所以输出会很随机 (Model is untrained, output will be random)\n")

# 生成 50 个 token
# 50個のトークンを生成します
with torch.no_grad():
    generated_ids = model.generate(
        context, 
        max_new_tokens=50,
        temperature=1.0,  # 随机性级别（randomness level）
        top_k=40,  # 每步采样前 40 个 token（sample from top 40 tokens）
    )

# 解码生成的文本
# 生成されたテキストをデコードします
generated_text = tokenizer.decode(generated_ids[0])
print("生成的文本 (Generated text):")
print(f"  {repr(generated_text)}")
print(f"\nToken 数: {len(generated_ids[0])}")

downloading https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json to C:\Users\zixun\.cache\mingpt\encoder.json
downloading https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe to C:\Users\zixun\.cache\mingpt\vocab.bpe
生成文本... (generating text...)
注意：模型还没训练，所以输出会很随机 (Model is untrained, output will be random)

生成的文本 (Generated text):
  '! SDiltrationInterview Cree Lois installing307WARNING enthusiasticoven ratiosrenchesDEFerrillaPsapped"> minions DVDs matte CircleESSION Torah SlugDuHope geop Refer calculateAc crou leve narrator eracycl experienced insertediery deathDef cute laureiferation 412Japanese Ish monks 172raftedonential'

Token 数: 51


## 第 6 步：模型内部调试工具
## Step 6: Model Internals Debugging

让我们使用 hooks 来观察模型内部发生的事情。

In [15]:
# 创建一个用于追踪中间激活值的字典
# 中間活性化を追跡するためのディクショナリを作成します
activations = {}

def make_hook(name):
    """创建一个 hook 函数来记录激活值"""
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations[name] = output.detach().cpu()
        elif isinstance(output, tuple) and len(output) > 0:
            # 如果输出是元组，取第一个张量元素
            # If output is tuple, take first tensor element
            if isinstance(output[0], torch.Tensor):
                activations[name] = output[0].detach().cpu()
    return hook

# 注册 hooks 到关键层
# 主要な層に フックを登録します
hooks = []
hooks.append(model.transformer.wte.register_forward_hook(make_hook('embedding')))
hooks.append(model.transformer.h[0].register_forward_hook(make_hook('first_block')))
if len(model.transformer.h) > 1:
    hooks.append(model.transformer.h[-1].register_forward_hook(make_hook('last_block')))
hooks.append(model.lm_head.register_forward_hook(make_hook('lm_head')))

# 前向传播
# 前向伝播を実行します
x_test = torch.randint(0, config.vocab_size, (1, 8)).to(device)
with torch.no_grad():
    _ = model(x_test)

# 打印激活值统计
# 活性化統計を印刷します
print("激活值统计 (Activation Statistics):")
print("="*60)
for name, activation in activations.items():
    print(f"\n{name}:")
    print(f"  形状 (Shape): {activation.shape}")
    print(f"  均值 (Mean): {activation.mean().item():.6f}")
    print(f"  标准差 (Std): {activation.std().item():.6f}")
    print(f"  最小值 (Min): {activation.min().item():.6f}")
    print(f"  最大值 (Max): {activation.max().item():.6f}")

# 移除 hooks
# フックを削除します
for hook in hooks:
    hook.remove()

激活值统计 (Activation Statistics):

embedding:
  形状 (Shape): torch.Size([1, 8, 48])
  均值 (Mean): -0.001262
  标准差 (Std): 0.019318
  最小值 (Min): -0.065372
  最大值 (Max): 0.051228

first_block:
  形状 (Shape): torch.Size([1, 8, 48])
  均值 (Mean): -0.000674
  标准差 (Std): 0.029932
  最小值 (Min): -0.104584
  最大值 (Max): 0.080235

last_block:
  形状 (Shape): torch.Size([1, 8, 48])
  均值 (Mean): -0.000117
  标准差 (Std): 0.033343
  最小值 (Min): -0.090414
  最大值 (Max): 0.084216

lm_head:
  形状 (Shape): torch.Size([1, 8, 50257])
  均值 (Mean): -0.000007
  标准差 (Std): 0.137867
  最小值 (Min): -0.634947
  最大值 (Max): 0.646422


## 第 7 步：训练一个简单的任务 (Optional)
## Step 7: Training on a Simple Task

现在让我们训练模型来学习一个简单的任务：序列重复模式。

In [16]:
from torch.utils.data import Dataset, DataLoader

class SimplePatternDataset(Dataset):
    """
    简单的模式数据集：给定一个模式，任务是预测下一个 token
    Simple pattern dataset: given a pattern, predict the next token
    
    例如：输入 [1,2,1,2] -> 输出应该是 3（下一个在这个简单序列中）
    Example: input [1,2,1,2] -> output should be 3 (next in sequence)
    """
    def __init__(self, num_samples=1000, seq_len=16, vocab_size=10):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # 创建简单的计数模式
        # 簡単なカウントパターンを作成します
        x = torch.arange(self.seq_len) % 5
        y = (x + 1) % 5  # 下一个数字（next number）
        return x, y

# 创建数据集和数据加载器
# データセットとデータローダーを作成します
train_dataset = SimplePatternDataset(num_samples=100, seq_len=8, vocab_size=5)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f"✓ 数据集创建完成 (Dataset created)")
print(f"  样本数 (Num samples): {len(train_dataset)}")
print(f"  批次大小 (Batch size): 8")
print(f"  序列长度 (Sequence length): 8")

# 检查一个样本
# サンプルを確認します
x_sample, y_sample = train_dataset[0]
print(f"\n样本示例 (Sample example):")
print(f"  输入 (Input): {x_sample.tolist()}")
print(f"  标签 (Target): {y_sample.tolist()}")

✓ 数据集创建完成 (Dataset created)
  样本数 (Num samples): 100
  批次大小 (Batch size): 8
  序列长度 (Sequence length): 8

样本示例 (Sample example):
  输入 (Input): [0, 1, 2, 3, 4, 0, 1, 2]
  标签 (Target): [1, 2, 3, 4, 0, 1, 2, 3]


## 第 8 步：使用 Trainer 进行训练
## Step 8: Training with Trainer

配置训练器并开始训练（这会需要几分钟）。

In [18]:
# 创建一个新的小模型用于训练
# トレーニング用に新しい小さいモデルを作成します
train_config = GPT.get_default_config()
train_config.model_type = 'gpt-nano'
train_config.device = device
train_config.vocab_size = 5  # 使用小词汇表（use small vocab）
train_config.block_size = 8

train_model = GPT(train_config)
train_model.to(device)

# 配置训练器
# トレーナーを設定します
trainer_config = Trainer.get_default_config()
trainer_config.device = device
trainer_config.num_workers = 0  # Windows 兼容性（Windows compatibility）
trainer_config.pin_memory = (device == 'cuda')  # 仅在有 GPU 时 pin memory（Only pin memory if GPU available）
trainer_config.max_iters = 50  # 只训练 50 步以节省时间（train for 50 steps to save time）
trainer_config.batch_size = 8
trainer_config.learning_rate = 6e-4
trainer_config.warmup_tokens = 0
trainer_config.final_tokens = trainer_config.max_iters * 8 * 8

# 创建训练器
# トレーナーを作成します
trainer = Trainer(trainer_config, train_model, train_dataset)

print(f"✓ 训练器配置完成 (Trainer configured)")
print(f"  最大迭代次数 (Max iterations): {trainer_config.max_iters}")
print(f"  批次大小 (Batch size): {trainer_config.batch_size}")
print(f"  学习率 (Learning rate): {trainer_config.learning_rate}")
print(f"  设备: {device}")
print(f"  Pin Memory: {trainer_config.pin_memory}")
print(f"\n开始训练... (starting training...)\n")

# 运行训练
# トレーニングを実行します
trainer.run()

number of parameters: 0.09M
running on device cpu
✓ 训练器配置完成 (Trainer configured)
  最大迭代次数 (Max iterations): 50
  批次大小 (Batch size): 8
  学习率 (Learning rate): 0.0006
  设备: cpu
  Pin Memory: False

开始训练... (starting training...)



## 第 9 步：在训练后生成文本
## Step 9: Generate After Training

现在让我们用训练后的模型生成文本，看看它是否学到了模式。

In [19]:
# 生成文本
# テキストを生成します
context = torch.tensor([[0, 1, 2, 3]]).to(device)  # 开始序列（start sequence）

print(f"开始序列 (Start sequence): {context[0].tolist()}")
print(f"\n生成文本（应该看到模式）...")
with torch.no_grad():
    generated = train_model.generate(context, max_new_tokens=16, temperature=0.1)

print(f"生成的序列 (Generated sequence): {generated[0].tolist()}")
print(f"\n分析 (Analysis):")
print(f"  模型在学习重复计数的模式吗？")
print(f"  Is the model learning the repeating counting pattern?")

开始序列 (Start sequence): [0, 1, 2, 3]

生成文本（应该看到模式）...
生成的序列 (Generated sequence): [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4]

分析 (Analysis):
  模型在学习重复计数的模式吗？
  Is the model learning the repeating counting pattern?


## 第 10 步：自动求导和计算图
## Step 10: Autograd and Computational Graph

让我们查看 PyTorch 的自动微分如何工作。

In [22]:
# 创建一个简单的前向传播和反向传播
# シンプルな前向伝播と逆伝播を作成します
config_for_grad = GPT.get_default_config()
config_for_grad.model_type = 'gpt-nano'
config_for_grad.vocab_size = 50257
config_for_grad.block_size = 1024
config_for_grad.device = device

model_for_grad = GPT(config_for_grad)
model_for_grad.to(device)

# 创建一个批次
# バッチを作成します
x = torch.randint(0, config_for_grad.vocab_size, (2, 8)).to(device)
y = torch.randint(0, config_for_grad.vocab_size, (2, 8)).to(device)

# 前向传播
# 前向伝播
output = model_for_grad(x)
logits = output[0] if isinstance(output, tuple) else output
loss = torch.nn.functional.cross_entropy(logits.view(-1, config_for_grad.vocab_size), y.view(-1))

print(f"损失 (Loss): {loss.item():.4f}")
print(f"\n计算图信息 (Computational Graph Info):")
print(f"  损失需要梯度 (Loss requires grad): {loss.requires_grad}")
print(f"  梯度函数 (Grad function): {loss.grad_fn}")

# 反向传播
# 逆伝播
loss.backward()

# 检查梯度
# 勾配を確認します
print(f"\n梯度统计 (Gradient Statistics):")
total_grad_norm = 0
for name, param in model_for_grad.named_parameters():
    if param.grad is not None:
        total_grad_norm += param.grad.data.norm(2).item() ** 2
        if 'wte' in name or 'lm_head' in name:
            print(f"  {name}: grad_norm={param.grad.data.norm(2).item():.6f}")

total_grad_norm = total_grad_norm ** 0.5
print(f"\n总梯度范数 (Total gradient norm): {total_grad_norm:.6f}")
print(f"  -> 用于梯度裁剪和学习率调度")
print(f"  -> Used for gradient clipping and learning rate scheduling")

number of parameters: 2.55M
损失 (Loss): 10.8034

计算图信息 (Computational Graph Info):
  损失需要梯度 (Loss requires grad): True
  梯度函数 (Grad function): <NllLossBackward0 object at 0x0000011A8C4A2530>

梯度统计 (Gradient Statistics):
  transformer.wte.weight: grad_norm=1.199692
  lm_head.weight: grad_norm=1.724503

总梯度范数 (Total gradient norm): 4.333503
  -> 用于梯度裁剪和学习率调度
  -> Used for gradient clipping and learning rate scheduling


## 总结
## Summary

你现在已经了解了：
1. ✓ 如何创建和配置 GPT 模型
2. ✓ 如何进行前向传播
3. ✓ 如何生成文本
4. ✓ 如何检查模型内部状态
5. ✓ 如何训练模型
6. ✓ 如何进行反向传播和梯度计算

---

## 下一步建议 (Next Steps)

1. **修改模型架构** - 改变层数、头数等
   Try changing n_layer, n_head, etc.

2. **运行完整的训练项目** - 尝试 projects/adder 或 projects/chargpt
   Run the full projects in projects/ folder

3. **分析注意力权重** - 看看模型"关注"什么
   Analyze what attention patterns the model learns

4. **使用预训练权重** - 加载 GPT-2 权重
   Load pretrained GPT-2 weights from HuggingFace

5. **自定义数据集** - 用你自己的数据训练
   Create custom datasets for specific tasks